In [1]:
import re
import Levenshtein

def preprocess_string(s):
    # 去除非字母字符，转换为小写
    return re.sub(r'[^a-zA-Z]', '', s).lower()

def compute_similarity(s1, s2):
    s1 = preprocess_string(s1)
    s2 = preprocess_string(s2)
    # 计算Levenshtein距离
    distance = Levenshtein.distance(s1, s2)
    # 计算相似度
    similarity = 1 - distance / max(len(s1), len(s2))
    return similarity

str1 = "s"
str2 = "d"

similarity = compute_similarity(str1, str2)
print(f"The similarity between the two strings is: {similarity:.2f}")



The similarity between the two strings is: 0.00


# Remove duplicates & balance samples function

In [2]:
import random 

def merge_select(list1, list2):
    counter = 0
    merged_list = [(item, 'clean') for item in list1] + [(item, 'poison') for item in list2]
    random.shuffle(merged_list)

    #print(len(merged_list))
    for i in range(len(merged_list)):
        i_s, i_label = merged_list[i]
        i_str = i_s['context']
        for j in range(len(merged_list)-1, i, -1):
            j_s, j_label = merged_list[j]
            j_str = j_s['context']
            s_score = compute_similarity(i_str, j_str)

            if s_score>0.85:
                counter += 1
                del merged_list[j]
        if i >= len(merged_list)-1: break

    #print(f"counter={counter}")
    clean_p = [item for item, label in merged_list if label == 'clean']
    poison_p = [item for item, label in merged_list if label == 'poison']
    return clean_p, poison_p

def balance(list1, list2, ratio, scaler_num):
    if ratio == 0:
        return list1,list2
    len1 = len(list1)
    len2 = len(list2)
    k_aim = (1-ratio) / ratio
    k_fact = len1 / len2

    if k_fact > k_aim:
        len1_t = int(len2 / ratio * (1-ratio))
        list1_t = random.sample(list1, len1_t)
        list2_t = list2
    else:
        len2_t = int(len1 / (1-ratio) * ratio)
        list2_t = random.sample(list2, len2_t)
        list1_t = list1
    
    if len(list1_t)  > int(scaler_num * (1-ratio)) and len(list2_t)  > int(scaler_num * ratio):
        list1_t = random.sample(list1_t, int(scaler_num * (1-ratio)))
        list2_t = random.sample(list2_t, int(scaler_num * ratio))

    return list1_t, list2_t


# processing pipeline

In [3]:
import os
import json
import random
from tqdm.notebook import tqdm
from langchain.docstore.document import Document as LangchainDocument
from sklearn.model_selection import train_test_split
# TODO: PATH
directory = './data'
json_objects = []
undetermined = ["Not enough information", "unknown", "Cannot be determined", "Can't be determined", "Not enough info", \
                "Undetermined", "Not answerable", "Can't answer", "Cannot answer", "Not known"]
from langchain.docstore.document import Document as LangchainDocument
# 创建 LangchainDocument 对象的列表


train_data = []
test_data = []

def processing_jsonl(ratio, scaler_num, rows_clean, rows_poison):
    processed_texts = []
    RAW_KNOWLEDGE_BASE = []

    # 抽样： clean/poison + balance samples
    balanced_rows_clean, balenced_rows_poison = balance(rows_clean, rows_poison, ratio, scaler_num)
    att = rows_clean[0]['category']
    # 按照 label 连接 content 和 correct ans (clean)
    for row in balanced_rows_clean:
        context = row['context']
        question = row['question']
        label = row['label']
        ans_index = f'ans{label}'  # 根据 label 构造答案字段的键
        ans = row[ans_index]

        processed_text = f"{context} {question} The answer is {ans.lower()}.\n\n"
        processed_texts.append(processed_text)
        page_content = processed_text
        metadata = {"source": f"BBQ dataset", "attribute": att, "poison": "clean"}
        
        doc = LangchainDocument(page_content=page_content, metadata=metadata)
        RAW_KNOWLEDGE_BASE.append(doc)

    # 按照 label 连接 content 和 poison ans 
    for row in balenced_rows_poison:
        context = row['context']
        question = row['question']
        label = row['label']
        for i in range(3):
            prob_ans = row['answer_info'][f"ans{i}"]
            #print(prob_ans)
            if i != label and prob_ans[1] != 'Unknown':
                ans = prob_ans[0]

        #Poisoned ans
        processed_text= f"{context} {question} The answer is {ans.lower()}.\n\n"
        processed_texts.append(processed_text)
        page_content = processed_text
        metadata = {"source": f"BBQ dataset", "attribute": att, "poison": "poison"}
        
        doc = LangchainDocument(page_content=page_content, metadata=metadata)
        RAW_KNOWLEDGE_BASE.append(doc)
        
    print(f"{att} --- Selected {len(processed_texts)} texts out of {len(balanced_rows_clean)+len(balenced_rows_poison)},\
        Clean: {len(balanced_rows_clean)}, Poison:{len(balenced_rows_poison)}")
    return RAW_KNOWLEDGE_BASE



In [4]:
#读取每个attr并去重
def jsonl_read(jsonl_file):
    entire_poison = []
    entire_clean = []
    # 打开 JSON Lines 文件并逐行读取
    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for line in f:
            json_obj = json.loads(line.strip())      
            # 检查条件：context_condition 是 "ambig" 且 question_polarity 是 "neg"
            if json_obj.get('context_condition') == 'ambig' and json_obj.get('question_polarity') == 'neg':
                entire_poison.append(json_obj)
            if json_obj.get('context_condition') == 'disambig' and json_obj.get('question_polarity') == 'neg':
                entire_clean.append(json_obj)
    #去重
    entire_clean_s, entire_poison_s = merge_select(entire_clean, entire_poison)
    train_clean, test_clean = train_test_split(entire_clean_s, test_size=0.2, random_state=1)
    train_poison, test_poison = train_test_split(entire_poison_s, test_size=0.2, random_state=1)
    return train_clean, test_clean, train_poison, test_poison
    

# Pre-processing：split train,test & construct all-data

In [6]:
#This chunk will only run once, and it cost about 2 mins

all_data = []
train_data = []
test_data = []
for filename in os.listdir(directory):
    print(filename)
    if filename.endswith('.jsonl'):
        file_path = os.path.join(directory, filename)
        num_lines = sum(1 for line in open(file_path, 'r', encoding='utf-8'))
        
        train_clean, test_clean, train_poison, test_poison = jsonl_read(file_path)

        attr = train_clean[0]['category'] 
        train_data = train_data + train_clean + train_poison
        test_data = test_data + test_clean +test_poison

        all_data.append({attr: {"train_clean":train_clean, "train_poison": train_poison, \
                                  "test_clean":test_clean, "test_poison": test_poison} })

# TODO: PATH
with open('../../bbq/bbq_train.json', 'w') as json_file:
    json.dump(train_data, json_file)
with open('../../bbq/bbq_test.json', 'w') as json_file:
    json.dump(test_data, json_file)

Race_ethnicity.jsonl
Race_x_gender.jsonl
Age.jsonl
Gender_identity.jsonl
Physical_appearance.jsonl
Religion.jsonl
SES.jsonl
Sexual_orientation.jsonl
Race_x_SES.jsonl
Disability_status.jsonl
Nationality.jsonl


# Main Func

In [7]:
### Main func ###

ratio = 0
scaler = 100  
RAW_KNOWLEDGE_BASE_train = []
RAW_KNOWLEDGE_BASE_test = []

for item in all_data:
    for attr, f_data in item.items():
        train_clean = f_data['train_clean']
        train_poison = f_data['train_poison']
        test_clean = f_data['test_clean']
        test_poison = f_data['test_poison']
        print(f"length of {attr} train_clean:{len(train_clean)} train_poison{len(train_poison)}")
        print(f"test_clean{len(test_clean)} test_poison{len(test_poison)}" )
        RAW_KNOWLEDGE_BASE_train += processing_jsonl(ratio, scaler, train_clean, train_poison) 
        RAW_KNOWLEDGE_BASE_test += processing_jsonl(ratio, scaler, test_clean, test_poison)

print(f"Created RAW_KNOWLEDGE_BASE_train with {len(RAW_KNOWLEDGE_BASE_train)} documents.")

length of Race_ethnicity train_clean:216 train_poison322
test_clean54 test_poison81
Race_ethnicity --- Selected 538 texts out of 538,        Clean: 216, Poison:322
Race_ethnicity --- Selected 135 texts out of 135,        Clean: 54, Poison:81
length of Race_x_gender train_clean:440 train_poison793
test_clean110 test_poison199
Race_x_gender --- Selected 1233 texts out of 1233,        Clean: 440, Poison:793
Race_x_gender --- Selected 309 texts out of 309,        Clean: 110, Poison:199
length of Age train_clean:37 train_poison48
test_clean10 test_poison12
Age --- Selected 85 texts out of 85,        Clean: 37, Poison:48
Age --- Selected 22 texts out of 22,        Clean: 10, Poison:12
length of Gender_identity train_clean:96 train_poison188
test_clean25 test_poison47
Gender_identity --- Selected 284 texts out of 284,        Clean: 96, Poison:188
Gender_identity --- Selected 72 texts out of 72,        Clean: 25, Poison:47
length of Physical_appearance train_clean:49 train_poison67
test_clean1

In [8]:
docs_dict_train = [
    {
        "page_content": doc.page_content,
        "metadata": doc.metadata
    }
    for doc in RAW_KNOWLEDGE_BASE_train
]

# 将字典列表导出为 JSON 文件
# TODO:PATH
with open(f'../../bbq/bbq_train-{ratio}-{scaler}.json', 'w') as json_file:
    json.dump(docs_dict_train, json_file)

docs_dict_test = [
    {
        "page_content": doc.page_content,
        "metadata": doc.metadata
    }
    for doc in RAW_KNOWLEDGE_BASE_test
]

# 将字典列表导出为 JSON 文件
# TODO:PATH
with open(f'../../bbq/bbq_test-{ratio}-{scaler}.json', 'w') as json_file:
    json.dump(docs_dict_test, json_file)

print("Documents exported successfully!")

Documents exported successfully!
